In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aadityakr5892/llm-classifer-medium/train.parquet
/kaggle/input/datasets/aadityakr5892/llm-classifer-medium/test.parquet
/kaggle/input/datasets/aadityakr5892/llm-classifer-medium/val.parquet
/kaggle/input/datasets/aadityakr5892/llm-classifier-dataset/train.parquet
/kaggle/input/datasets/aadityakr5892/llm-classifier-dataset/test.parquet
/kaggle/input/datasets/aadityakr5892/llm-classifier-dataset/val.parquet


In [2]:
!pip install -q transformers datasets torchmetrics

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torchmetrics import Accuracy
from tqdm.auto import tqdm
import os
import gc

# Address Kaggle timeout/saving issues by ensuring we save to the working directory
SAVE_DIR = '/kaggle/working/models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# 1. Load your pre-split datasets using read_parquet
train_df = pd.read_parquet('/kaggle/input/datasets/aadityakr5892/llm-classifer-medium/train.parquet')
val_df = pd.read_parquet('/kaggle/input/datasets/aadityakr5892/llm-classifer-medium/val.parquet')
test_df = pd.read_parquet('/kaggle/input/datasets/aadityakr5892/llm-classifer-medium/test.parquet')

# 2. Encode Labels (Fit ONLY on train, transform all)
label_encoder = LabelEncoder()
train_df['label'] = label_encoder.fit_transform(train_df['generated_by'])
val_df['label'] = label_encoder.transform(val_df['generated_by'])
test_df['label'] = label_encoder.transform(test_df['generated_by'])

NUM_CLASSES = len(label_encoder.classes_)
print(f"Classes found ({NUM_CLASSES}): {label_encoder.classes_}")

# 3. Extract to lists for the DataLoader
train_texts = train_df['text'].tolist()
train_labels = train_df['label'].tolist()

val_texts = val_df['text'].tolist()
val_labels = val_df['label'].tolist()

test_texts = test_df['text'].tolist()
test_labels = test_df['label'].tolist()

print(f"Train size: {len(train_texts)} | Val size: {len(val_texts)} | Test size: {len(test_texts)}")

Classes found (12): ['chatgpt' 'cohere' 'cohere-chat' 'gpt2' 'gpt3' 'gpt4' 'human'
 'llama-chat' 'mistral' 'mistral-chat' 'mpt' 'mpt-chat']
Train size: 79681 | Val size: 9959 | Test size: 9959


In [5]:
TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

# TUNED: Increased to 384 based on p90 text length stat (381)
MAX_LEN = 384
BATCH_SIZE = 64 

class LLMDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = LLMDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = LLMDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = LLMDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
class AttentionMechanism(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)
        
    def forward(self, rnn_outputs, mask):
        attn_scores = self.attention(rnn_outputs).squeeze(-1) 
        attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_weights = F.softmax(attn_scores, dim=1).unsqueeze(-1) 
        context_vector = torch.sum(attn_weights * rnn_outputs, dim=1) 
        return context_vector, attn_weights

class AttentionBiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, drop_prob=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_token_id)
        
        self.gru = nn.GRU(
            embed_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            bidirectional=True, 
            batch_first=True, 
            dropout=drop_prob if num_layers > 1 else 0
        )
        
        self.attention = AttentionMechanism(hidden_dim * 2)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(drop_prob),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, input_ids, attention_mask):
        embedded = self.embedding(input_ids) 
        gru_out, _ = self.gru(embedded) 
        context, _ = self.attention(gru_out, attention_mask)
        logits = self.classifier(context)
        return logits

# Initialize Model
VOCAB_SIZE = tokenizer.vocab_size
EMBED_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.4

model = AttentionBiGRU(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, NUM_CLASSES, DROPOUT)
model = model.to(device)
print(model)

AttentionBiGRU(
  (embedding): Embedding(30522, 256, padding_idx=0)
  (gru): GRU(256, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (attention): AttentionMechanism(
    (attention): Linear(in_features=512, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=256, out_features=12, bias=True)
  )
)


In [7]:
EPOCHS = 8
# TUNED: Slightly higher LR for larger sequence lengths
LEARNING_RATE = 3e-4 

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
accuracy_metric = Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(device)

In [8]:
best_val_acc = 0.0

for epoch in range(EPOCHS):
    print(f"\n{'='*30}")
    print(f"         EPOCH {epoch+1}/{EPOCHS}")
    print(f"{'='*30}")
    
    # --- TRAINING ---
    model.train()
    train_loss_accum = 0
    accuracy_metric.reset()
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
        optimizer.step()
        
        train_loss_accum += loss.item()
        accuracy_metric.update(logits, labels)
        
        # Optional: Print progress every 500 batches to avoid Kaggle log freeze
        if batch_idx > 0 and batch_idx % 500 == 0:
            print(f"  [Train] Batch {batch_idx}/{len(train_loader)}")
            
    train_acc = accuracy_metric.compute().item()
    epoch_train_loss = train_loss_accum / len(train_loader)
    
    # --- VALIDATION ---
    model.eval()
    val_loss_accum = 0
    accuracy_metric.reset()
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            val_loss_accum += loss.item()
            accuracy_metric.update(logits, labels)
            
    val_acc = accuracy_metric.compute().item()
    epoch_val_loss = val_loss_accum / len(val_loader)
    
    scheduler.step()
    
    # --- OUTPUT ALL 4 PARAMETERS ---
    print("\n  📊 Epoch Results:")
    print(f"  Training Accuracy:   {train_acc:.4f}  |  Training Loss:   {epoch_train_loss:.4f}")
    print(f"  Validation Accuracy: {val_acc:.4f}  |  Validation Loss: {epoch_val_loss:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_path = os.path.join(SAVE_DIR, 'best_bigru_attention.pth')
        torch.save(model.state_dict(), save_path)
        print(f"\n  ⭐ New best validation accuracy! Model saved.")
        
    gc.collect()
    torch.cuda.empty_cache()


         EPOCH 1/8
  [Train] Batch 500/1246
  [Train] Batch 1000/1246

  📊 Epoch Results:
  Training Accuracy:   0.4105  |  Training Loss:   1.6883
  Validation Accuracy: 0.5178  |  Validation Loss: 1.3658

  ⭐ New best validation accuracy! Model saved.

         EPOCH 2/8
  [Train] Batch 500/1246
  [Train] Batch 1000/1246

  📊 Epoch Results:
  Training Accuracy:   0.5638  |  Training Loss:   1.2556
  Validation Accuracy: 0.5889  |  Validation Loss: 1.1918

  ⭐ New best validation accuracy! Model saved.

         EPOCH 3/8
  [Train] Batch 500/1246
  [Train] Batch 1000/1246

  📊 Epoch Results:
  Training Accuracy:   0.6340  |  Training Loss:   1.0582
  Validation Accuracy: 0.6232  |  Validation Loss: 1.0762

  ⭐ New best validation accuracy! Model saved.

         EPOCH 4/8
  [Train] Batch 500/1246
  [Train] Batch 1000/1246

  📊 Epoch Results:
  Training Accuracy:   0.6868  |  Training Loss:   0.9050
  Validation Accuracy: 0.6518  |  Validation Loss: 1.0225

  ⭐ New best validation acc